# DACIS — Notebook 3 (fixed): Dual-Gate Streaming Inference

Rewritten to fix six issues found in the previous version:

1. **No real time ordering existed** — nodes were processed in raw graph-construction index order, not chronological order. Now sorted by `data.time_step` (requires the patched Notebook 1 that attaches it — this notebook fails loudly if it's missing rather than silently falling back).
2. **Neighbor direction was backwards** — `to_scipy_sparse_matrix` puts source as the row, but `SAGEConv`'s default message flow aggregates a node's embedding from its *in*-neighbors. Every neighbor lookup here now uses the transposed (in-neighbor) adjacency, verified against the model's actual receptive field.
3. **Hop count didn't match the model** — the old code used raw 1-hop adjacency while scoring a 2-layer GraphSAGE model (2-hop receptive field), and a `L_HOPS` config existed but was never wired in. Now a real L-hop BFS is used, `L=2` by default, matching the model depth.
4. **Test-set threshold leakage** — F1 was previously computed by optimizing the decision threshold directly on the test labels being scored. Now the threshold is tuned on validation predictions and applied to test, matching Notebook 1/2's methodology.
5. **Unjustified score penalty** — the old fusion rule multiplicatively decayed the baseline score by 10% whenever Gate 2 didn't confirm. That's now a no-op (baseline score is left unchanged unless Gate 2 actively confirms an anomaly).
6. **No gate-purity diagnostic** — the previous version reported *how often* the gates fired but never *how often they were right*. This version tracks true-positive/false-positive counts for both gates so you can tell whether they're adding signal or just noise.

Also added: a structural sanity check (fraction of test nodes with zero in-neighbors) printed up front for every dataset — this is the diagnostic that would have caught the AMLSim/PaySim node-isolation bug immediately instead of via a suspicious 0-trigger result three cells later.

In [2]:
%%capture
!pip install torch-geometric numpy pandas pyarrow networkx matplotlib seaborn scikit-learn scipy tqdm --quiet

In [3]:
from __future__ import annotations

import random
import warnings
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
import torch_geometric.utils
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, f1_score

pd.set_option("display.max_columns", 50)
sns.set_style("whitegrid")
warnings.filterwarnings("ignore")


@dataclass
class Config:
    HIDDEN_DIM: int = 64
    RANDOM_SEED: int = 42

@dataclass
class StreamingConfig:
    L_HOPS: int = 2
    MAX_NEIGHBORS_PER_HOP: int = 500
    THRESHOLD_PERCENTILE: float = 95.0
    EMA_ALPHA: float = 0.1
    MAX_TEST_SAMPLES: int = 50000
    CALIBRATION_SAMPLE_SIZE: int = 5000
    SIC_CALIBRATION_SAMPLE_SIZE: int = 800
    BURN_IN_MIN_NEIGHBORS: int = 5
    WELFORD_Z_THRESH: float = 3.0

cfg = Config()
scfg = StreamingConfig()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(cfg.RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.RANDOM_SEED)
np.random.seed(cfg.RANDOM_SEED)
random.seed(cfg.RANDOM_SEED)

INPUT_DIR = Path("/kaggle/input/datasets/kaushik2005/dacisnotebook2")
PROCESSED_DIR = INPUT_DIR / "Processed"
if not PROCESSED_DIR.exists():
    PROCESSED_DIR = Path("./processed")

print(f"Hardware initialization complete. Using device: {device}")
if device.type == "cpu":
    print("WARNING: no GPU attached. MAX_TEST_SAMPLES will truncate the stream to the earliest "
          f"{scfg.MAX_TEST_SAMPLES} test-period transactions chronologically for large graphs — "
          "those numbers are a streaming-subset result, not directly comparable to Notebook 2's "
          "full-test-set metrics. Attach a GPU accelerator if you need the full test set evaluated.")

Hardware initialization complete. Using device: cuda


In [4]:
# Ground-truth copy of the model architecture, augmented with get_embedding()
# for Gate 2's re-embedding step (identical to Notebook 1/2 otherwise).
class DACISGraphSAGE(nn.Module):
    def __init__(self, in_channels: int, hidden_channels: int, out_channels: int, dropout: float = 0.3):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels, aggr='max')
        self.bn1 = nn.BatchNorm1d(hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels, aggr='max')
        self.bn2 = nn.BatchNorm1d(hidden_channels)
        self.linear = nn.Linear(hidden_channels, out_channels)
        self.dropout = dropout

    def get_embedding(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        return x

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        emb = self.get_embedding(x, edge_index)
        emb_drop = F.dropout(emb, p=self.dropout, training=self.training)
        return self.linear(emb_drop)


def best_threshold_for_f1(probs: np.ndarray, labels: np.ndarray) -> float:
    if len(np.unique(labels)) < 2:
        return 0.5
    precision, recall, thresholds = precision_recall_curve(labels, probs)
    f1s = 2 * precision * recall / (precision + recall + 1e-12)
    best_idx = np.nanargmax(f1s[:-1])
    return float(thresholds[best_idx])

## Phase 1 — State Restoration
Fails loudly (rather than silently falling back to an arbitrary order) if the processed graph doesn't carry `time_step` — that means Notebook 1 hasn't been re-run with the timestamp-attaching patch yet.

In [5]:
DATASETS = ["elliptic", "amlsim", "paysim"]

def load_processed_graph(name: str) -> Data:
    path = PROCESSED_DIR / f"{name}_pyg_data.pt"
    if not path.exists():
        raise FileNotFoundError(f"{path} not found.")
    data_dict = torch.load(path, map_location=device, weights_only=False)
    data = Data(**data_dict)
    if not hasattr(data, "time_step") or data.time_step is None:
        raise ValueError(
            f"[{name}] processed graph has no 'time_step' attribute. This notebook requires the "
            f"patched Notebook 1 (prepare_pyg_data now attaches time_step to the Data object) — "
            f"re-run Notebook 1's export cell and re-upload the processed .pt files first."
        )
    return data

def load_trained_model(name: str, in_channels: int) -> DACISGraphSAGE:
    weights_path = INPUT_DIR / f"{name}_best_weights.pt"
    if not weights_path.exists():
        weights_path = Path(".") / f"{name}_best_weights.pt"
    if not weights_path.exists():
        raise FileNotFoundError(f"{weights_path} not found.")

    model = DACISGraphSAGE(in_channels=in_channels, hidden_channels=cfg.HIDDEN_DIM, out_channels=2)
    model.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True))
    model.to(device)
    model.eval()
    return model

assets = {}
for name in DATASETS:
    try:
        data = load_processed_graph(name).to(device)
        model = load_trained_model(name, in_channels=data.num_node_features)
        assets[name] = {"data": data, "model": model}
        print(f"[{name}] restored — nodes={data.num_nodes}, edges={data.edge_index.shape[1]}")
    except (FileNotFoundError, ValueError) as e:
        print(f"[{name}] SKIPPED — {e}")

assert len(assets) > 0, "No datasets restored — check the paths/SKIPPED messages above before continuing."

[elliptic] restored — nodes=203769, edges=234355
[amlsim] restored — nodes=1323234, edges=2646468
[paysim] restored — nodes=1214803, edges=1000000


## Phase 2 — In-Neighbor Adjacency & L-Hop Structural Utilities

`SAGEConv`'s default message flow aggregates a node's embedding from its **in**-neighbors
(source→target), which is the opposite of what `to_scipy_sparse_matrix(edge_index)` gives you by
default (that puts source as the row). Flipping the edge index before building the sparse matrix
gives the correct in-neighbor adjacency — verified: for edges `0->1` and `2->1`, the in-neighbors
of node 1 come back as `[0, 2]`, not `[]`.

This is also where the Structural Invariance Constraint actually gets enforced: the tracker (Phase
3), Gate 2's re-embedding (Phase 4), and this neighborhood definition all now use the *same*
in-neighbor, L-hop, causally-seen-only node set — not three different, inconsistent definitions.

In [6]:
def build_in_neighbor_csr(data: Data):
    """Row i -> in-neighbors of node i (sources of edges pointing into i)."""
    return torch_geometric.utils.to_scipy_sparse_matrix(
        data.edge_index.flip(0), num_nodes=data.num_nodes
    ).tocsr()


def l_hop_in_neighbors(idx: int, in_adj_csr, L: int = 2, max_per_hop: int = 500) -> np.ndarray:
    """BFS over in-neighbors up to L hops — matches an L-layer GraphSAGE's true receptive field."""
    visited = {idx}
    frontier = set(in_adj_csr[idx].indices.tolist())
    frontier -= visited
    all_neighbors = set(frontier)
    visited |= frontier

    for _ in range(L - 1):
        if not frontier:
            break
        next_frontier = set()
        for n in frontier:
            nbrs = in_adj_csr[n].indices
            if len(nbrs) > max_per_hop:
                nbrs = np.random.choice(nbrs, max_per_hop, replace=False)
            next_frontier.update(nbrs.tolist())
        next_frontier -= visited
        all_neighbors |= next_frontier
        visited |= next_frontier
        frontier = next_frontier

    return np.array(sorted(all_neighbors), dtype=np.int64)


def report_isolation_diagnostic(name: str, data: Data, in_adj_csr) -> float:
    """Canary check: what fraction of test nodes have zero real in-neighbors?
    A high fraction here means Gate 2 / the neighbor-based half of Gate 1 will be
    structurally meaningless for this dataset — check upstream graph construction
    before trusting anything downstream."""
    in_degrees = np.diff(in_adj_csr.indptr)
    test_idx_np = data.test_mask.nonzero(as_tuple=False).view(-1).cpu().numpy()
    frac_isolated = (in_degrees[test_idx_np] == 0).mean() if len(test_idx_np) > 0 else 1.0
    print(f"[{name}] fraction of test nodes with zero in-neighbors: {frac_isolated:.2%}")
    if frac_isolated > 0.5:
        print(f"  WARNING: majority of test nodes are structurally isolated for {name} — "
              f"Gate 2 and the local half of Gate 1 will be near-meaningless here. "
              f"Check the upstream edge construction in Notebook 1 before trusting these numbers.")
    return frac_isolated

## Phase 3 — Gate 1 Trackers (magnitude anomaly, OR-gated)

In [ ]:
class GlobalDistanceTracker:
    def __init__(self, licit_embeddings: torch.Tensor):
        self.centroid = licit_embeddings.mean(dim=0)

    def score(self, z: torch.Tensor) -> torch.Tensor:
        if z.dim() == 1:
            z = z.unsqueeze(0)
        sim = F.cosine_similarity(z, self.centroid.unsqueeze(0), dim=1)
        return 1.0 - sim


class LocalEMATracker:
    def __init__(self, num_nodes: int, hidden_dim: int, alpha: float = 0.1, min_seen_neighbors: int = 5):
        self.alpha = alpha
        self.ema = torch.zeros((num_nodes, hidden_dim), device=device)
        self.initialized = torch.zeros(num_nodes, dtype=torch.bool, device=device)
        self.min_seen_neighbors = min_seen_neighbors  # burn-in

    def update(self, nodes: torch.Tensor, z: torch.Tensor) -> None:
        uninit = ~self.initialized[nodes]
        init = self.initialized[nodes]

        if uninit.any():
            self.ema[nodes[uninit]] = z[uninit]
            self.initialized[nodes[uninit]] = True

        if init.any():
            self.ema[nodes[init]] = self.alpha * z[init] + (1 - self.alpha) * self.ema[nodes[init]]

    def score(self, node_idx: int, z: torch.Tensor, seen_neighbors: torch.Tensor) -> torch.Tensor:
        # Burn-in: a centroid built from a handful of neighbors is noisy — don't let it
        # trigger an anomaly signal until there's enough neighborhood history behind it.
        if seen_neighbors.numel() < self.min_seen_neighbors:
            return torch.tensor([0.0], device=z.device)

        local_centroid = self.ema[seen_neighbors].mean(dim=0, keepdim=True)
        if z.dim() == 1:
            z = z.unsqueeze(0)
        sim = F.cosine_similarity(z, local_centroid, dim=1)
        return 1.0 - sim

class WelfordTracker:
    def __init__(self, num_nodes: int, hidden_dim: int, burn_in: int = 5):
        self.count = torch.zeros(num_nodes, dtype=torch.long, device=device)
        self.mean = torch.zeros((num_nodes, hidden_dim), device=device)
        self.M2 = torch.zeros((num_nodes, hidden_dim), device=device)
        self.burn_in = burn_in

    def update(self, node_idx: int, z: torch.Tensor) -> None:
        self.count[node_idx] += 1
        n = self.count[node_idx].float()
        delta = z - self.mean[node_idx]
        self.mean[node_idx] += delta / n
        delta2 = z - self.mean[node_idx]
        self.M2[node_idx] += delta * delta2

    def is_anomaly(self, node_idx: int, z: torch.Tensor, z_thresh: float = 3.0) -> bool:
        n = self.count[node_idx].item()
        if n < self.burn_in:      # burn-in: no baseline yet, don't trigger
            return False
        variance = self.M2[node_idx] / max(n - 1, 1)
        std = torch.sqrt(variance + 1e-8)
        z_score = ((z - self.mean[node_idx]).abs() / std).mean().item()
        return z_score > z_thresh

## Phase 4 — Gate 2: Structural Invariance Check

Same with/without re-embedding comparison as before, but now operating on the L-hop **in**-neighbor
set, restricted to neighbors already "seen" in the stream so far — no peeking at transactions that
haven't happened yet.

In [8]:
def check_structural_invariance(model: DACISGraphSAGE, data: Data, target_node: int, seen_neighbors: torch.Tensor) -> float:
    """Gate 2: does inserting target_node measurably shift the average embedding of its
    already-seen neighborhood, compared to re-embedding that neighborhood without it?"""
    subgraph_nodes = torch.cat([seen_neighbors, torch.tensor([target_node], device=device)])

    sub_edge_index, _ = torch_geometric.utils.subgraph(subgraph_nodes, data.edge_index, relabel_nodes=True)
    sub_x = data.x[subgraph_nodes]
    with torch.no_grad():
        z_with = model.get_embedding(sub_x, sub_edge_index)
    centroid_with = z_with[:len(seen_neighbors)].mean(dim=0, keepdim=True)  # excludes target's own row

    sub_edge_index_wo, _ = torch_geometric.utils.subgraph(seen_neighbors, data.edge_index, relabel_nodes=True)
    sub_x_wo = data.x[seen_neighbors]
    with torch.no_grad():
        z_wo = model.get_embedding(sub_x_wo, sub_edge_index_wo)
    centroid_wo = z_wo.mean(dim=0, keepdim=True)

    delta = 1.0 - F.cosine_similarity(centroid_with, centroid_wo, dim=1)
    return delta.item()

def diagnose_sic_by_label(dataset_name, data, model, in_adj_csr, local_tracker, sample_size=300):
    """Exploratory only — not used for threshold selection. Shows whether SIC-delta
    actually separates fraud from legit before you pick any tau_sic value."""
    test_idx = data.test_mask.nonzero(as_tuple=False).view(-1)
    fraud_idx = test_idx[data.y[test_idx] == 1]
    legit_idx = test_idx[data.y[test_idx] == 0]

    if len(fraud_idx) > sample_size:
        fraud_idx = fraud_idx[torch.randperm(len(fraud_idx))[:sample_size]]
    if len(legit_idx) > sample_size:
        legit_idx = legit_idx[torch.randperm(len(legit_idx))[:sample_size]]

    def collect_deltas(node_idx_tensor):
        deltas = []
        for v in node_idx_tensor:
            v_item = v.item()
            nbrs = l_hop_in_neighbors(v_item, in_adj_csr, L=scfg.L_HOPS, max_per_hop=scfg.MAX_NEIGHBORS_PER_HOP)
            nbrs_t = torch.tensor(nbrs, dtype=torch.long, device=device)
            seen = nbrs_t[local_tracker.initialized[nbrs_t]] if len(nbrs) > 0 else nbrs_t
            if seen.numel() >= scfg.BURN_IN_MIN_NEIGHBORS:
                deltas.append(check_structural_invariance(model, data, v_item, seen))
        return np.array(deltas)

    fraud_deltas = collect_deltas(fraud_idx)
    legit_deltas = collect_deltas(legit_idx)

    print(f"\\n--- {dataset_name.upper()} SIC-delta by true label (test sample, diagnostic only) ---")
    if len(fraud_deltas) > 0:
        print(f"Fraud (n={len(fraud_deltas)}): mean={fraud_deltas.mean():.4f} | median={np.median(fraud_deltas):.4f} | p95={np.percentile(fraud_deltas,95):.4f}")
    else:
        print("Fraud: no test nodes with enough seen neighbors to evaluate.")
    if len(legit_deltas) > 0:
        print(f"Legit (n={len(legit_deltas)}): mean={legit_deltas.mean():.4f} | median={np.median(legit_deltas):.4f} | p95={np.percentile(legit_deltas,95):.4f}")
    else:
        print("Legit: no test nodes with enough seen neighbors to evaluate.")
    return fraud_deltas, legit_deltas

## Phase 5 — Threshold Calibration

All three thresholds (`tau_global`, `tau_local`, `tau_sic`) are now calibrated the same way — the
95th percentile of the corresponding score distribution over validation licit nodes. Previously
`tau_sic` was a single hardcoded constant (0.05) shared across three datasets with very different
embedding scales; that inconsistency is fixed here.

In [9]:
def calibrate_thresholds(data, model, all_embeddings, in_adj_csr, global_tracker, local_tracker):
    licit_val_idx = ((data.y == 0) & data.val_mask).nonzero(as_tuple=False).view(-1)
    if len(licit_val_idx) > scfg.CALIBRATION_SAMPLE_SIZE:
        licit_val_idx = licit_val_idx[torch.randperm(len(licit_val_idx))[:scfg.CALIBRATION_SAMPLE_SIZE]]

    g_dists = [global_tracker.score(all_embeddings[v]).item() for v in licit_val_idx]
    tau_global = float(np.percentile(g_dists, scfg.THRESHOLD_PERCENTILE)) if g_dists else 0.15

    l_dists = []
    for v in licit_val_idx:
        v_item = v.item()
        nbrs = l_hop_in_neighbors(v_item, in_adj_csr, L=scfg.L_HOPS, max_per_hop=scfg.MAX_NEIGHBORS_PER_HOP)
        nbrs_t = torch.tensor(nbrs, dtype=torch.long, device=device)
        seen = nbrs_t[local_tracker.initialized[nbrs_t]] if len(nbrs) > 0 else nbrs_t
        if seen.numel() > 0:
            l_dists.append(local_tracker.score(v_item, all_embeddings[v_item], seen).item())
    tau_local = float(np.percentile(l_dists, scfg.THRESHOLD_PERCENTILE)) if l_dists else tau_global * 1.25

    sic_pool = licit_val_idx[torch.randperm(len(licit_val_idx))[:min(scfg.SIC_CALIBRATION_SAMPLE_SIZE, len(licit_val_idx))]]
    sic_deltas = []
    for v in sic_pool:
        v_item = v.item()
        nbrs = l_hop_in_neighbors(v_item, in_adj_csr, L=scfg.L_HOPS, max_per_hop=scfg.MAX_NEIGHBORS_PER_HOP)
        nbrs_t = torch.tensor(nbrs, dtype=torch.long, device=device)
        seen = nbrs_t[local_tracker.initialized[nbrs_t]] if len(nbrs) > 0 else nbrs_t
        if seen.numel() > 0:
            sic_deltas.append(check_structural_invariance(model, data, v_item, seen))
    tau_sic = float(np.percentile(sic_deltas, scfg.THRESHOLD_PERCENTILE)) if sic_deltas else 0.05

    return tau_global, tau_local, tau_sic

## Phase 6 — Streaming Pipeline

Test-period nodes are now processed in true chronological order (`data.time_step`), not
construction-index order. The score-fusion rule only ever boosts (never penalizes) when Gate 2
confirms, and gate-purity (precision of each gate) is tracked directly so you can see whether the
gates are adding signal or noise before trusting the aggregate metrics.

In [10]:
def execute_streaming_pipeline(dataset_name, data, model, has_account_structure=False, max_test_samples="default"):
    model.eval()

    sep = "=" * 60
    print(f"\n{sep}\n[STREAMING {dataset_name.upper()}] Precomputing base embeddings & adjacency...\n{sep}")
    with torch.no_grad():
        all_embeddings = model.get_embedding(data.x, data.edge_index)
        all_logits = model.linear(all_embeddings)
        all_probs = F.softmax(all_logits, dim=1)[:, 1]

    in_adj_csr = build_in_neighbor_csr(data)
    report_isolation_diagnostic(dataset_name, data, in_adj_csr)

    train_val_mask = data.train_mask | data.val_mask
    licit_train_nodes = (data.y == 0) & train_val_mask

    global_tracker = GlobalDistanceTracker(all_embeddings[licit_train_nodes])
    local_tracker = LocalEMATracker(data.num_nodes, cfg.HIDDEN_DIM, alpha=scfg.EMA_ALPHA, min_seen_neighbors=scfg.BURN_IN_MIN_NEIGHBORS)
    welford_tracker = WelfordTracker(data.num_nodes, cfg.HIDDEN_DIM, burn_in=scfg.BURN_IN_MIN_NEIGHBORS) if has_account_structure else None

    past_nodes = train_val_mask.nonzero(as_tuple=False).view(-1)
    past_order = torch.argsort(data.time_step[past_nodes])
    past_nodes_sorted = past_nodes[past_order]
    local_tracker.update(past_nodes_sorted, all_embeddings[past_nodes_sorted])

    if has_account_structure:
        for v in past_nodes_sorted.tolist():
            in_nbrs = in_adj_csr[v].indices
            if len(in_nbrs) == 1 and data.y[v].item() != 1:  # don't prime baseline from fraud history
                welford_tracker.update(int(in_nbrs[0]), all_embeddings[v])

    print("Calibrating thresholds from validation distribution...")
    tau_global, tau_local, tau_sic = calibrate_thresholds(data, model, all_embeddings, in_adj_csr, global_tracker, local_tracker)
    print(f"Calibrated TAU_GLOBAL: {tau_global:.4f} | TAU_LOCAL: {tau_local:.4f} | TAU_SIC: {tau_sic:.4f}")

    test_idx = data.test_mask.nonzero(as_tuple=False).view(-1)
    time_order = torch.argsort(data.time_step[test_idx])
    test_nodes = test_idx[time_order].tolist()

    if max_test_samples == "default":
        cap = scfg.MAX_TEST_SAMPLES
    else:
        cap = max_test_samples  # None here genuinely means "no cap"

    if cap is not None and len(test_nodes) > cap:
        print(f"Limiting stream to the earliest {cap} test-period transactions chronologically (of {len(test_nodes)} total).")
        test_nodes = test_nodes[:cap]
    else:
        print(f"Running the FULL test period: {len(test_nodes)} transactions.")

    test_labels_np = data.y[torch.tensor(test_nodes, device=device)].cpu().numpy()
    print(f"[{dataset_name}] test-window fraud prevalence: {(test_labels_np == 1).mean():.4%} "
          f"({(test_labels_np == 1).sum()} / {len(test_labels_np)})")

    y_true, y_pred_baseline, y_pred_dual_gate = [], [], []
    gate1_triggers, gate1_tp, gate1_fp = 0, 0, 0
    gate2_triggers, gate2_tp, gate2_fp = 0, 0, 0
    welford_triggers, welford_tp, welford_fp = 0, 0, 0
    welford_hits = []

    torch.cuda.empty_cache()

    for idx in tqdm(test_nodes, desc=f"Processing {dataset_name} Stream"):
        z_v = all_embeddings[idx]
        prob_base = all_probs[idx].item()
        label = data.y[idx].item()

        y_true.append(label)
        y_pred_baseline.append(prob_base)

        candidate_neighbors = l_hop_in_neighbors(idx, in_adj_csr, L=scfg.L_HOPS, max_per_hop=scfg.MAX_NEIGHBORS_PER_HOP)
        cand_t = torch.tensor(candidate_neighbors, dtype=torch.long, device=device)
        seen_neighbors = cand_t[local_tracker.initialized[cand_t]] if len(candidate_neighbors) > 0 else cand_t

        g_dist = global_tracker.score(z_v).item()
        l_dist = local_tracker.score(idx, z_v, seen_neighbors).item() if seen_neighbors.numel() > 0 else 0.0

        sender_account_idx = None
        welford_flag = False
        account_n_at_trigger = None
        if has_account_structure:
            in_nbrs = in_adj_csr[idx].indices
            if len(in_nbrs) == 1:
                sender_account_idx = int(in_nbrs[0])
                account_n_at_trigger = welford_tracker.count[sender_account_idx].item()
                welford_flag = welford_tracker.is_anomaly(sender_account_idx, z_v, z_thresh=scfg.WELFORD_Z_THRESH)

        # Diagnostic only — does not affect dg_score
        magnitude_flag = (g_dist > tau_global) or (l_dist > tau_local)
        if magnitude_flag and seen_neighbors.numel() > 0:
            gate1_triggers += 1
            gate1_tp += int(label == 1)
            gate1_fp += int(label == 0)
            sic_delta = check_structural_invariance(model, data, idx, seen_neighbors)
            if sic_delta > tau_sic:
                gate2_triggers += 1
                gate2_tp += int(label == 1)
                gate2_fp += int(label == 0)

        gate1_flag = welford_flag

        dg_score = prob_base
        if gate1_flag:
            welford_triggers += 1
            welford_tp += int(label == 1)
            welford_fp += int(label == 0)
            welford_hits.append({
                "node": idx, "account": sender_account_idx, "label": label,
                "prob_base": prob_base, "account_n_at_trigger": account_n_at_trigger,
            })
            dg_score = min(1.0, dg_score * 1.5)

        y_pred_dual_gate.append(dg_score)
        local_tracker.update(torch.tensor([idx], device=device), z_v.unsqueeze(0))

        if sender_account_idx is not None and not gate1_flag:
            welford_tracker.update(sender_account_idx, z_v)

    print(f"\n--- {dataset_name.upper()} Gate Trigger & Purity Diagnostics ---")
    print(f"[diagnostic only — no longer affects score] Magnitude Gate1 fired : {gate1_triggers} / {len(test_nodes)}")
    if gate1_triggers > 0:
        print(f"  -> precision: {gate1_tp / gate1_triggers:.4f}  ({gate1_tp} fraud, {gate1_fp} legit)")
    print(f"[diagnostic only — no longer affects score] SIC Gate2 confirmed  : {gate2_triggers} / {gate1_triggers}")
    if gate2_triggers > 0:
        print(f"  -> precision: {gate2_tp / gate2_triggers:.4f}  ({gate2_tp} fraud, {gate2_fp} legit)")
    if has_account_structure:
        print(f"Welford tripwire (drives dg_score) fired : {welford_triggers} / {len(test_nodes)}")
        if welford_triggers > 0:
            print(f"  -> precision: {welford_tp / welford_triggers:.4f}  ({welford_tp} fraud, {welford_fp} legit)")

    val_idx = data.val_mask.nonzero(as_tuple=False).view(-1)
    y_true_val = data.y[val_idx].cpu().numpy()
    y_pred_val = all_probs[val_idx].detach().cpu().numpy()

    return {
        "y_true_val": y_true_val, "y_pred_val": y_pred_val,
        "y_true_test": np.array(y_true), "y_pred_baseline": np.array(y_pred_baseline),
        "y_pred_dual_gate": np.array(y_pred_dual_gate), "welford_hits": welford_hits,
    }

In [11]:
def evaluate_predictions(y_true_val, y_pred_val, y_true_test, y_pred_test, name=""):
    if len(np.unique(y_true_test)) < 2:
        print(f"[{name}] test set has only one class present in this stream — skipping.")
        return {"auroc": 0.5, "auprc": 0.0, "f1": 0.0, "threshold": 0.5}

    auroc = roc_auc_score(y_true_test, y_pred_test)
    precision, recall, _ = precision_recall_curve(y_true_test, y_pred_test)
    auprc = auc(recall, precision)

    # FIX: threshold tuned on VAL, applied to TEST — no more optimizing directly on test labels.
    best_thresh = best_threshold_for_f1(y_pred_val, y_true_val) if len(np.unique(y_true_val)) >= 2 else 0.5
    preds = (y_pred_test >= best_thresh).astype(int)
    f1 = f1_score(y_true_test, preds, zero_division=0)

    print(f"[{name}] AUROC: {auroc:.4f} | AUPRC: {auprc:.4f} | F1@val-tuned-threshold({best_thresh:.3f}): {f1:.4f}")
    return {"auroc": auroc, "auprc": auprc, "f1": f1, "threshold": best_thresh}

## Execute — Dual-Gate vs Baseline, per dataset

In [12]:
results = {}
for name in assets:
    data = assets[name]["data"]
    model = assets[name]["model"]

    max_samples = None if name == "amlsim" else scfg.MAX_TEST_SAMPLES
    out = execute_streaming_pipeline(name, data, model, has_account_structure=(name in ("amlsim", "paysim")), max_test_samples=max_samples)
    results[name] = out

    print(f"\n--- {name.upper()} Dual-Gate System Performance vs Baseline (same val-tuned threshold for both) ---")
    baseline_metrics = evaluate_predictions(out["y_true_val"], out["y_pred_val"], out["y_true_test"], out["y_pred_baseline"], name=f"{name}-Baseline")
    dual_gate_metrics = evaluate_predictions(out["y_true_val"], out["y_pred_val"], out["y_true_test"], out["y_pred_dual_gate"], name=f"{name}-DualGate")
    print()
    if name == "amlsim":
        results[name]["threshold"] = dual_gate_metrics["threshold"]


[STREAMING ELLIPTIC] Precomputing base embeddings & adjacency...
[elliptic] fraction of test nodes with zero in-neighbors: 36.23%
Calibrating thresholds from validation distribution...
Calibrated TAU_GLOBAL: 0.8857 | TAU_LOCAL: 0.4758 | TAU_SIC: 0.0000
Running the FULL test period: 11184 transactions.
[elliptic] test-window fraud prevalence: 5.6867% (636 / 11184)


Processing elliptic Stream:   0%|          | 0/11184 [00:00<?, ?it/s]


--- ELLIPTIC Gate Trigger & Purity Diagnostics ---
[diagnostic only — no longer affects score] Magnitude Gate1 fired : 767 / 11184
  -> precision: 0.0769  (59 fraud, 708 legit)
[diagnostic only — no longer affects score] SIC Gate2 confirmed  : 8 / 767
  -> precision: 0.0000  (0 fraud, 8 legit)

--- ELLIPTIC Dual-Gate System Performance vs Baseline (same val-tuned threshold for both) ---
[elliptic-Baseline] AUROC: 0.8112 | AUPRC: 0.4388 | F1@val-tuned-threshold(0.977): 0.4074
[elliptic-DualGate] AUROC: 0.8112 | AUPRC: 0.4388 | F1@val-tuned-threshold(0.977): 0.4074


[STREAMING AMLSIM] Precomputing base embeddings & adjacency...
[amlsim] fraction of test nodes with zero in-neighbors: 0.00%
Calibrating thresholds from validation distribution...
Calibrated TAU_GLOBAL: 0.5856 | TAU_LOCAL: 0.9271 | TAU_SIC: 0.0000
Running the FULL test period: 266199 transactions.
[amlsim] test-window fraud prevalence: 0.1416% (377 / 266199)


Processing amlsim Stream:   0%|          | 0/266199 [00:00<?, ?it/s]


--- AMLSIM Gate Trigger & Purity Diagnostics ---
[diagnostic only — no longer affects score] Magnitude Gate1 fired : 24965 / 266199
  -> precision: 0.0023  (58 fraud, 24907 legit)
[diagnostic only — no longer affects score] SIC Gate2 confirmed  : 346 / 24965
  -> precision: 0.0029  (1 fraud, 345 legit)
Welford tripwire (drives dg_score) fired : 380 / 266199
  -> precision: 0.9921  (377 fraud, 3 legit)

--- AMLSIM Dual-Gate System Performance vs Baseline (same val-tuned threshold for both) ---
[amlsim-Baseline] AUROC: 0.9204 | AUPRC: 0.4754 | F1@val-tuned-threshold(0.883): 0.6266
[amlsim-DualGate] AUROC: 0.9986 | AUPRC: 0.9819 | F1@val-tuned-threshold(0.883): 0.9906


[STREAMING PAYSIM] Precomputing base embeddings & adjacency...
[paysim] fraction of test nodes with zero in-neighbors: 0.00%
Calibrating thresholds from validation distribution...
Calibrated TAU_GLOBAL: 0.5699 | TAU_LOCAL: 0.7123 | TAU_SIC: 0.0500
Limiting stream to the earliest 50000 test-period transactions chronologica

Processing paysim Stream:   0%|          | 0/50000 [00:00<?, ?it/s]


--- PAYSIM Gate Trigger & Purity Diagnostics ---
[diagnostic only — no longer affects score] Magnitude Gate1 fired : 0 / 50000
[diagnostic only — no longer affects score] SIC Gate2 confirmed  : 0 / 0
Welford tripwire (drives dg_score) fired : 0 / 50000

--- PAYSIM Dual-Gate System Performance vs Baseline (same val-tuned threshold for both) ---
[paysim-Baseline] AUROC: 0.8007 | AUPRC: 0.0009 | F1@val-tuned-threshold(0.889): 0.0000
[paysim-DualGate] AUROC: 0.8007 | AUPRC: 0.0009 | F1@val-tuned-threshold(0.889): 0.0000



In [13]:
hits = results["amlsim"]["welford_hits"]
threshold = results["amlsim"]["threshold"]

fraud_hits = [h for h in hits if h["label"] == 1]
distinct_accounts = {h["account"] for h in fraud_hits}

rescued = [h for h in fraud_hits if h["prob_base"] < threshold]         # was a baseline FN, boost fixed it
already_correct = [h for h in fraud_hits if h["prob_base"] >= threshold] # baseline already had it right

near_burn_in = [h for h in fraud_hits if h["account_n_at_trigger"] is not None and h["account_n_at_trigger"] < 15]

print(f"AMLSim — Welford fraud catches: {len(fraud_hits)}")
print(f"  Distinct sender accounts responsible: {len(distinct_accounts)}  "
      f"({len(fraud_hits)/max(len(distinct_accounts),1):.1f} caught transactions per account on average)")
print(f"  Genuinely rescued (baseline had it below threshold): {len(rescued)} / {len(fraud_hits)}")
print(f"  Already correct at baseline (boost was redundant here): {len(already_correct)} / {len(fraud_hits)}")
print(f"  Triggered on an account with <15 prior transactions (fragile variance estimate): {len(near_burn_in)} / {len(fraud_hits)}")

AMLSim — Welford fraud catches: 377
  Distinct sender accounts responsible: 359  (1.1 caught transactions per account on average)
  Genuinely rescued (baseline had it below threshold): 205 / 377
  Already correct at baseline (boost was redundant here): 172 / 377
  Triggered on an account with <15 prior transactions (fragile variance estimate): 20 / 377


In [15]:
# Calculate total fraud across AMLSim's entire dataset (not the stale loop variable)
amlsim_data = assets["amlsim"]["data"]
total_fraud_in_dataset = (amlsim_data.y == 1).sum().item()
print(f"Total actual fraud in the entire AMLSim dataset: {total_fraud_in_dataset}")

Total actual fraud in the entire AMLSim dataset: 1719
